# Classification - Option B (Without BFPME Score Variables)

Option B removes BFPME score variables to test a more data-driven model.

Models used:
- RandomForestClassifier
- XGBoost (or GradientBoosting fallback)

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

In [ ]:
_cwd = Path.cwd().resolve()
project_root = _cwd.parent if _cwd.name == "notebook" else _cwd
DATA_DIR = project_root / "data"
data_path = DATA_DIR / "final_dataset.csv"
df = pd.read_csv(data_path)
target_col = "default_flag"

# Business feature engineering aligned with data_preprocessing_feature_engineering.ipynb
df["debt_total"] = df["dettes_court_terme"] + df["dettes_long_terme"]
df["loan_to_revenue"] = df["montant_pret"] / df["chiffre_affaires_annuel"].replace(0, np.nan)
df["cash_to_debt"] = df["tresorerie_disponible"] / df["debt_total"].replace(0, np.nan)
df["financement_externe_intensity"] = df["ratio_financement_externe_projet"]
df["impayes_to_loan"] = df["montant_impayes"] / df["montant_pret"].replace(0, np.nan)
df["payment_stress_index"] = (
    np.clip(df["retards_paiement_jours_moyen"] / 90, 0, 3)
    + np.clip(df["impayes_to_loan"], 0, 3)
)
df["fdr_bfr_pressure"] = df["besoin_fonds_roulement_bfr"] / df["fonds_roulement_fdr"].replace(0, np.nan)
df["garantie_to_loan"] = df["valeur_garanties"] / df["montant_pret"].replace(0, np.nan)
df = df.replace([np.inf, -np.inf], np.nan)

# Option B: remove BFPME score-related variables.
bfpme_cols = [
    "risk_class", "bfpme_risk_index",
    "score_bfpme_good", "score_bfpme_average", "score_bfpme_global",
    "score_bfpme_global_operational", "score_bfpme_gap",
    "score_financier", "score_business", "score_management", "score_secteur_industrie",
    "score_global_interne",
] + [c for c in df.columns if c.startswith("SF_")]
drop_cols = ["dossier_id", "pd_target"] + bfpme_cols
drop_cols = [c for c in drop_cols if c in df.columns]

X = df.drop(columns=drop_cols + [target_col])
y = df[target_col].astype(int)
X.shape

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\BFPME\\data\\dataset_final_realism_v2.csv'

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

ordinal_map = {
    "duree_credit": ["court_terme", "moyen_terme", "long_terme"],
}
ordinal_features = [c for c in ordinal_map if c in X_train.columns]
ordinal_categories = [ordinal_map[c] for c in ordinal_features]

nominal_features = [
    "secteur_activite", "region_localisation", "structure_juridique",
    "classification_pme", "zone_localisation", "type_credit", "objectif_financement",
    "type_taux", "scenario_economique",
]
nominal_features = [c for c in nominal_features if c in X_train.columns]

numeric_features = [c for c in X_train.select_dtypes(include=[np.number]).columns if c not in ordinal_features]

winsor_cols = [
    "montant_pret", "montant_total_investissement", "valeur_garanties", "chiffre_affaires_annuel",
    "total_actif", "total_passif", "capitaux_propres", "dettes_court_terme", "dettes_long_terme",
    "fonds_roulement_fdr", "besoin_fonds_roulement_bfr", "tresorerie_disponible", "montant_impayes",
    "loan_to_revenue", "cash_to_debt", "impayes_to_loan", "fdr_bfr_pressure", "garantie_to_loan",
]
winsor_numeric_features = [c for c in winsor_cols if c in numeric_features]
plain_numeric_features = [c for c in numeric_features if c not in winsor_numeric_features]

class Winsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, lower_quantile=0.01, upper_quantile=0.99):
        self.lower_quantile = lower_quantile
        self.upper_quantile = upper_quantile
    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.lower_bounds_ = X.quantile(self.lower_quantile)
        self.upper_bounds_ = X.quantile(self.upper_quantile)
        return self
    def transform(self, X):
        X = pd.DataFrame(X).copy()
        return X.clip(self.lower_bounds_, self.upper_bounds_, axis=1).values

preprocessor = ColumnTransformer(transformers=[
    ("winsor_num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("winsorizer", Winsorizer()), ("scaler", StandardScaler())]), winsor_numeric_features),
    ("plain_num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), plain_numeric_features),
    ("ord", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OrdinalEncoder(categories=ordinal_categories, handle_unknown="use_encoded_value", unknown_value=-1))]), ordinal_features),
    ("nom", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), nominal_features),
])

X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)
X_train_p.shape, X_test_p.shape

((8000, 177), (2000, 177))

In [4]:
# final_dataset.csv has no temporal date; models use the stratified random split above.
print("Train/Test shapes:", X_train.shape, X_test.shape)

Temporal v2 shapes: (8003, 92) (1997, 92)


In [5]:
def evaluate_model(model_name, model, X_train_p, y_train, X_test_p, y_test):
    model.fit(X_train_p, y_train)
    y_pred = model.predict(X_test_p)
    y_prob = model.predict_proba(X_test_p)[:, 1] if hasattr(model, "predict_proba") else y_pred
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "auc": roc_auc_score(y_test, y_prob),
    }, confusion_matrix(y_test, y_pred)

rf_model = RandomForestClassifier(n_estimators=400, min_samples_split=8, min_samples_leaf=4, random_state=42, n_jobs=-1)

if HAS_XGB:
    xgb_model = XGBClassifier(
        n_estimators=500, max_depth=5, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85,
        objective="binary:logistic", eval_metric="auc", random_state=42
    )
    xgb_name = "XGBoost"
else:
    xgb_model = GradientBoostingClassifier(random_state=42)
    xgb_name = "GradientBoosting_fallback"

results, cms = [], {}
for name, model in [("RandomForest", rf_model), (xgb_name, xgb_model)]:
    res, cm = evaluate_model(name, model, X_train_p, y_train, X_test_p, y_test)
    results.append(res)
    cms[name] = cm

pd.DataFrame(results).sort_values("auc", ascending=False)

,model,accuracy,precision,recall,f1,auc
0,RandomForest,0.800701,0.840521,0.836437,0.838474,0.834103
1,XGBoost,0.789685,0.822644,0.841296,0.831865,0.824017


In [6]:
for model_name, cm in cms.items():
    print(f"\n{model_name} - Confusion Matrix")
    print(cm)


RandomForest - Confusion Matrix
[[ 566  196]
 [ 202 1033]]

XGBoost - Confusion Matrix
[[ 538  224]
 [ 196 1039]]
